# Lab 4B — CNN for Text Classification

### Aim
Implement a Convolutional Neural Network (CNN) for text classification using word embeddings and compare its performance with an LSTM model.

### Dataset
IMDb Movie Reviews:
- `0` → Negative
- `1` → Positive

## Program Flow

```text
IMDb Reviews
     ↓
Integer Sequences
     ↓
Padding
     ↓
Embedding
     ↓
CNN
     ↓
Sentiment
```

We will then build an LSTM using the same data and compare the accuracy.

## 1. Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

## 2. Load the IMDb Dataset

In [2]:
vocab_size = 10000
max_length = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 25000
Testing samples: 25000


## 3. Pad the Sequences

Reviews have different lengths. Padding makes every review the same length.

```text
Different lengths
       ↓
Same length = 200
```

In [3]:
X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding="post"
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (25000, 200)
Testing shape: (25000, 200)


## 4. Build the CNN Model

The **Embedding** layer converts word numbers into vectors.

The **Conv1D** layer looks for useful local patterns in the text.

The **GlobalMaxPooling1D** layer keeps the strongest detected feature.

In [4]:
cnn = Sequential([
    Embedding(vocab_size, 32, input_length=max_length),
    Conv1D(32, 3, activation="relu"),
    GlobalMaxPooling1D(),
    Dense(1, activation="sigmoid")
])

cnn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn.summary()

/Users/karthibalasundaram/Downloads/deep_Learning_DSATM/.venv-tf/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 5. Train the CNN

In [5]:
cnn_history = cnn.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)

Epoch 1/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7098 - loss: 0.5952 - val_accuracy: 0.7832 - val_loss: 0.4592
Epoch 2/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8322 - loss: 0.3824 - val_accuracy: 0.8440 - val_loss: 0.3692
Epoch 3/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8805 - loss: 0.2897 - val_accuracy: 0.8680 - val_loss: 0.3205
Epoch 4/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9166 - loss: 0.2205 - val_accuracy: 0.8756 - val_loss: 0.3054
Epoch 5/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9449 - loss: 0.1658 - val_accuracy: 0.8732 - val_loss: 0.3085


## 6. Evaluate the CNN

In [6]:
cnn_loss, cnn_accuracy = cnn.evaluate(X_test, y_test, verbose=0)

print("CNN Test Accuracy:", round(cnn_accuracy * 100, 2), "%")

CNN Test Accuracy: 86.87 %


## 7. Build the LSTM Model

Now we use an LSTM instead of CNN.

```text
Embedding → LSTM → Dense → Output
```

The LSTM processes the text as a sequence.

In [7]:
lstm = Sequential([
    Embedding(vocab_size, 32, input_length=max_length),
    LSTM(32),
    Dense(1, activation="sigmoid")
])

lstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 8. Train the LSTM

In [8]:
lstm_history = lstm.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)

Epoch 1/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 11s 58ms/step - accuracy: 0.5558 - loss: 0.6799 - val_accuracy: 0.6360 - val_loss: 0.6625
Epoch 2/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - accuracy: 0.5992 - loss: 0.6485 - val_accuracy: 0.6068 - val_loss: 0.6430
Epoch 3/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - accuracy: 0.6708 - loss: 0.5625 - val_accuracy: 0.8244 - val_loss: 0.4526
Epoch 4/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 11s 60ms/step - accuracy: 0.8412 - loss: 0.4069 - val_accuracy: 0.8364 - val_loss: 0.4232
Epoch 5/5
176/176 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - accuracy: 0.8482 - loss: 0.3958 - val_accuracy: 0.8204 - val_loss: 0.4518


## 9. Evaluate the LSTM

In [9]:
lstm_loss, lstm_accuracy = lstm.evaluate(X_test, y_test, verbose=0)

print("LSTM Test Accuracy:", round(lstm_accuracy * 100, 2), "%")

LSTM Test Accuracy: 80.25 %


## 10. Compare CNN and LSTM

In [10]:
print("----- CNN vs LSTM -----")
print("CNN  Accuracy :", round(cnn_accuracy * 100, 2), "%")
print("LSTM Accuracy :", round(lstm_accuracy * 100, 2), "%")

if cnn_accuracy > lstm_accuracy:
    print("CNN performed better in this experiment.")
elif lstm_accuracy > cnn_accuracy:
    print("LSTM performed better in this experiment.")
else:
    print("Both models achieved the same accuracy.")

----- CNN vs LSTM -----
CNN  Accuracy : 86.87 %
LSTM Accuracy : 80.25 %
CNN performed better in this experiment.


# Result

The CNN and LSTM models were successfully implemented for IMDb sentiment classification.

### Key Difference

**CNN:** Detects important local patterns in text.

**LSTM:** Processes text sequentially and maintains information from previous words.

The better model should be determined from the actual accuracy obtained during the experiment.

# Viva Questions

1. What is text classification?
2. What is the IMDb dataset?
3. Why do we use padding?
4. What is an embedding layer?
5. What does Conv1D do?
6. Why is GlobalMaxPooling1D used?
7. What is an LSTM?
8. What is the difference between CNN and LSTM?
9. Why do we use sigmoid in the output layer?
10. Which model performed better in your experiment?